# Final Serialized Pipeline / Model — Fixed Salary Deduplication

This notebook is the reproducible source of truth for the fixed final model artifact. It starts from the salary-deduplicated processed train/test files, loads the tuned Gradient Boosting model selected during the fixed tuning stage, refits it on the full 2021–2024 training set, evaluates the untouched 2025 holdout set, and saves the final model bundle plus final results.

Inputs:
- `data/processed_salary_fix/X_train_processed.csv`
- `data/processed_salary_fix/y_train.csv`
- `data/processed_salary_fix/X_test_2025_processed.csv`
- `data/processed_salary_fix/y_test_2025.csv`
- `data/processed_salary_fix/player_lookup_train.csv`
- `data/processed_salary_fix/player_lookup_test_2025.csv`
- `artifacts/Fixed/best_model_fixed.joblib`

Outputs:
- `artifacts/Fixed/final_model_fixed.joblib`
- `results/Fixed/final fixed/final_metrics_fixed.csv`
- `results/Fixed/final fixed/final_predictions_fixed.csv`
- `results/Fixed/final fixed/final_model_metadata_fixed.json`

The fixed version uses the salary de-duplication rule developed during the audit: each player keeps one salary record, using the highest available salary as the practical full-season salary proxy when contract dates are unavailable.


In [6]:
import json
import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# Locate project root robustly, whether this notebook is run from /notebooks,
# /notebooks/Deduplicated, or the repo root.
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Fixed salary-deduplication pipeline paths
processed_dir = ROOT / "data" / "processed_salary_fix"
artifact_dir = ROOT / "artifacts" / "Fixed"
final_result_dir = ROOT / "results" / "Fixed" / "final fixed"

final_result_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

# Tuned Gradient Boosting model saved by model_tuning.ipynb under the fixed pipeline
tuned_model_path = artifact_dir / "best_model_fixed.joblib"

RANDOM_STATE = 26
TRAIN_YEARS = [2021, 2022, 2023, 2024]
TEST_YEAR = 2025
FINAL_MODEL_NAME = "GradientBoostingRegressor"
RESIDUAL_DEFINITION = "predicted - actual"

print("ROOT:", ROOT)
print("processed_dir:", processed_dir)
print("artifact_dir:", artifact_dir)
print("final_result_dir:", final_result_dir)
print("tuned_model_path:", tuned_model_path)
print("tuned model exists:", tuned_model_path.exists())

ROOT: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation
processed_dir: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\data\processed_salary_fix
artifact_dir: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\artifacts\Fixed
final_result_dir: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\final fixed
tuned_model_path: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\artifacts\Fixed\best_model_fixed.joblib
tuned model exists: True


### Load Fixed Processed Train/Test Data

This section loads the processed artifacts generated by `train_test_processed.ipynb` using the fixed salary de-duplication logic. The feature matrix, target vector, and player lookup tables are checked for shape alignment before fitting the final model.


In [7]:
X_train = pd.read_csv(processed_dir / "X_train_processed.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]

X_test = pd.read_csv(processed_dir / "X_test_2025_processed.csv")
y_test = pd.read_csv(processed_dir / "y_test_2025.csv")["salary"]

lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")
lookup_test = pd.read_csv(processed_dir / "player_lookup_test_2025.csv")

feature_names = pd.read_csv(processed_dir / "feature_names.csv")["feature"].tolist()

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(f"Train/test feature count mismatch: {X_train.shape[1]} vs {X_test.shape[1]}")

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError("Train/test feature columns are not aligned in the same order.")

if X_train.columns.tolist() != feature_names:
    raise ValueError("X_train columns do not match feature_names.csv.")

if len(X_train) != len(y_train) or len(X_train) != len(lookup_train):
    raise ValueError("X_train, y_train, and lookup_train row counts do not match.")

if len(X_test) != len(y_test) or len(X_test) != len(lookup_test):
    raise ValueError("X_test, y_test, and lookup_test row counts do not match.")

if lookup_test["player"].duplicated().sum() != 0:
    raise ValueError("Duplicate players remain in the 2025 lookup table.")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of final features:", len(feature_names))
print("Duplicated 2025 lookup players:", lookup_test["player"].duplicated().sum())

display(pd.DataFrame({"feature": feature_names}).head(30))

X_train shape: (633, 26)
X_test shape: (182, 26)
Number of final features: 26
Duplicated 2025 lookup players: 0


,feature
0,avail_rate
1,blk
2,fg
3,fg_per_g
4,fga
5,ft
6,fta
7,g
8,mp
9,pca1


### Load and Refit Final Gradient Boosting Model

The final model is the tuned Gradient Boosting model selected by the fixed tuning stage using cross-validated RMSE. The saved tuned estimator is loaded from `artifacts/Fixed/best_model_fixed.joblib` and refit on the full 2021–2024 training set before evaluating the untouched 2025 holdout set.

`RANDOM_STATE = 26` is retained as the project-wide reproducibility seed; it does not imply that the model is Random Forest.


In [8]:
if not tuned_model_path.exists():
    raise FileNotFoundError(
        f"Missing tuned fixed model: {tuned_model_path}. "
        "Run model_tuning.ipynb with the fixed paths first."
    )

loaded_model = joblib.load(tuned_model_path)

# If a bundle/dict was saved accidentally, extract the estimator. Otherwise use the estimator directly.
if isinstance(loaded_model, dict):
    if "model" not in loaded_model:
        raise KeyError("Loaded model bundle is a dict but does not contain a 'model' key.")
    final_model = loaded_model["model"]
else:
    final_model = loaded_model

if type(final_model).__name__ != FINAL_MODEL_NAME:
    raise TypeError(
        f"Expected {FINAL_MODEL_NAME}, but loaded {type(final_model).__name__}. "
        "Check that best_model_fixed.joblib comes from the fixed Gradient Boosting tuning run."
    )

# Refit the selected tuned estimator on the full 2021-2024 training set.
final_model.fit(X_train, y_train)

print("Final model:", final_model)
print("Model type:", type(final_model).__name__)
print("Model parameters:")

display(pd.Series(final_model.get_params()))

Final model: GradientBoostingRegressor(learning_rate=0.05, max_depth=2, min_samples_leaf=3,
                          random_state=26)
Model type: GradientBoostingRegressor
Model parameters:


alpha                                 0.9
ccp_alpha                             0.0
criterion                    friedman_mse
init                                 None
learning_rate                        0.05
loss                        squared_error
max_depth                               2
max_features                         None
max_leaf_nodes                       None
min_impurity_decrease                 0.0
min_samples_leaf                        3
min_samples_split                       2
min_weight_fraction_leaf              0.0
n_estimators                          100
n_iter_no_change                     None
random_state                           26
subsample                             1.0
tol                                0.0001
validation_fraction                   0.1
verbose                                 0
warm_start                          False
dtype: object

### Holdout Prediction and Evaluation

This section applies the final Gradient Boosting model to the untouched 2025 holdout set. It creates player-level predictions with actual salary, predicted salary, residual, and absolute error, then computes the final KPI metrics.

Residuals are defined as `predicted - actual`. Positive residuals indicate the model predicted a higher salary than the player actually earned, while negative residuals indicate the model predicted a lower salary than the player actually earned.


In [9]:
# Generate holdout salary predictions for the untouched 2025 test set
pred = final_model.predict(X_test)

# Build a player-level prediction table with identifiers, actual salary, predicted salary, and error columns
pred_df = lookup_test.copy()
pred_df["actual"] = y_test.to_numpy()
pred_df["predicted"] = pred
pred_df["residual"] = pred_df["predicted"] - pred_df["actual"]
pred_df["abs_error"] = pred_df["residual"].abs()

# Keep the final prediction output columns in a clear reporting order
pred_cols = ["player", "team", "year", "group", "actual", "predicted", "residual", "abs_error"]
pred_df = pred_df[pred_cols]

if pred_df["player"].duplicated().sum() != 0:
    raise ValueError("Duplicate players appear in final prediction output.")

# Compute final holdout KPIs and store split/model context for traceability
metrics = {
    "model": type(final_model).__name__,
    "train_years": "2021-2024",
    "test_year": TEST_YEAR,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(len(feature_names)),
    "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
    "MAE": float(mean_absolute_error(y_test, pred)),
    "MAPE": float(mean_absolute_percentage_error(y_test, pred)),
    "R2": float(r2_score(y_test, pred)),
    "residual_definition": RESIDUAL_DEFINITION,
}

metrics_df = pd.DataFrame([metrics])

display(metrics_df)
display(pred_df.sort_values("abs_error", ascending=False).head(10))

,model,train_years,test_year,n_train,n_test,n_features,RMSE,MAE,MAPE,R2,residual_definition
0,GradientBoostingRegressor,2021-2024,2025,633,182,26,39717.19788,30368.066572,0.695309,0.631117,predicted - actual


,player,team,year,group,actual,predicted,residual,abs_error
128,Moriah Jefferson,CHI,2025,veteran,145500,31196.673240,-114303.326760,114303.326760
70,Jewell Loyd,LVA,2025,veteran,249032,137885.228522,-111146.771478,111146.771478
152,Sabrina Ionescu,NYL,2025,rookie,222060,112676.942368,-109383.057632,109383.057632
18,Arike Ogunbowale,DAL,2025,unknown,249032,143333.767414,-105698.232586,105698.232586
173,Teaira McCowan,DAL,2025,veteran,201400,96696.534160,-104703.465840,104703.465840
11,Alysha Clark,TOT,2025,veteran,205908,112233.508095,-93674.491905,93674.491905
13,Amy Okonkwo,DAL,2025,veteran,7774,100097.832115,92323.832115,92323.832115
77,Kaila Charles,TOT,2025,veteran,21198,112318.451879,91120.451879,91120.451879
141,Odyssey Sims,TOT,2025,veteran,54622,141170.915845,86548.915845,86548.915845
58,Haley Jones,TOT,2025,veteran,36094,119276.601018,83182.601018,83182.601018


### Save Final Results

This section saves the fixed final KPI table and the fixed player-level prediction table. These outputs should be used by the final visualizations and executive summary for the fixed pipeline.


In [10]:
metrics_path = final_result_dir / "final_metrics_fixed.csv"
predictions_path = final_result_dir / "final_predictions_fixed.csv"

metrics_df.to_csv(metrics_path, index=False)
pred_df.to_csv(predictions_path, index=False)

print("Saved final metrics to:", metrics_path)
print("Saved final predictions to:", predictions_path)

Saved final metrics to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\final fixed\final_metrics_fixed.csv
Saved final predictions to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\final fixed\final_predictions_fixed.csv


### Save Final Model Bundle

This section saves the final refit Gradient Boosting model as a reusable `joblib` bundle and saves a separate JSON metadata file for inspection.

The `final_model_fixed.joblib` bundle stores the fitted model, feature names, model parameters, metrics, split information, residual definition, and metadata. The JSON file records the same information in a human-readable format.


In [11]:
split_info = {
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
}

metadata = {
    "artifact_name": "final_model_fixed.joblib",
    "model_type": type(final_model).__name__,
    "model_params": final_model.get_params(),
    "feature_names": feature_names,
    "target": "salary",
    "split_info": split_info,
    "metrics": metrics,
    "residual_definition": RESIDUAL_DEFINITION,
    "pipeline_version": "salary_deduplicated_fixed",
    "input_files": {
        "X_train": str(processed_dir / "X_train_processed.csv"),
        "y_train": str(processed_dir / "y_train.csv"),
        "X_test": str(processed_dir / "X_test_2025_processed.csv"),
        "y_test": str(processed_dir / "y_test_2025.csv"),
        "lookup_train": str(processed_dir / "player_lookup_train.csv"),
        "lookup_test": str(processed_dir / "player_lookup_test_2025.csv"),
        "tuned_model": str(tuned_model_path),
    },
    "output_files": {
        "model_bundle": str(artifact_dir / "final_model_fixed.joblib"),
        "metrics": str(metrics_path),
        "predictions": str(predictions_path),
        "metadata": str(final_result_dir / "final_model_metadata.json"),
    },
    "environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
        "joblib_version": joblib.__version__,
    },
}

model_bundle = {
    "model": final_model,
    "feature_names": feature_names,
    "model_params": final_model.get_params(),
    "metrics": metrics,
    "split_info": split_info,
    "target": "salary",
    "residual_definition": RESIDUAL_DEFINITION,
    "metadata": metadata,
}

model_path = artifact_dir / "final_model_fixed.joblib"
metadata_path = final_result_dir / "final_model_metadata_fixed.json"

joblib.dump(model_bundle, model_path)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved final model bundle to:", model_path)
print("Saved final metadata to:", metadata_path)

Saved final model bundle to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\artifacts\Fixed\final_model_fixed.joblib
Saved final metadata to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\final fixed\final_model_metadata_fixed.json
